In [20]:
# ============================================================
# Cellule 1 — Imports et chargement des fichiers
# ============================================================
import json
import csv
import re
from pathlib import Path

JSON_PATH = "..\List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio.json"   # <- ton export Label Studio
CSV_PATH  = "..\List-of-images\JJ096-JJ099_image_data.csv"                # <- ta liste complète d'images
OUTPUT_PATH = "..\List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio_completed.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print(f"Nombre de tâches dans le JSON : {len(tasks)}")
print("Exemple de tâche :")
print(json.dumps(tasks[0], indent=2, ensure_ascii=False)[:1000])

# ============================================================
# Cellule 2 — Repérer le champ qui contient le nom/chemin de l'image
# ============================================================
# Dans un export Label Studio classique, chaque tâche a la forme :
# { "id": ..., "data": {"image": "..."}, "annotations": [...], ... }
# Adapte le nom de la clé si besoin (ex: "image", "img", "ocr", etc.)

IMAGE_FIELD = "image_path"

print(tasks[0]["data"].keys())

Nombre de tâches dans le JSON : 1547
Exemple de tâche :
{
  "id": 42918,
  "annotations": [
    {
      "id": 42918,
      "completed_by": 1,
      "result": [
        {
          "id": "4_NIA_0.9905",
          "type": "rectanglelabels",
          "value": {
            "x": 0.305,
            "y": 0.02,
            "width": 99.53,
            "height": 99.94,
            "rotation": 0,
            "rectanglelabels": [
              "NIA"
            ]
          },
          "to_name": "image",
          "from_name": "label",
          "image_rotation": 0,
          "original_width": 2048,
          "original_height": 2048
        }
      ],
      "was_cancelled": false,
      "ground_truth": true,
      "created_at": "2026-06-22T15:39:01.949497Z",
      "updated_at": "2026-06-22T15:39:01.949497Z",
      "draft_created_at": null,
      "lead_time": null,
      "prediction": {},
      "result_count": 0,
      "unique_id": "effd3ba4-05fa-4300-a943-22a1305cc59b",
      "import_id": 3095,

<>:9: SyntaxWarning: invalid escape sequence '\L'
<>:10: SyntaxWarning: invalid escape sequence '\L'
<>:11: SyntaxWarning: invalid escape sequence '\L'
<>:9: SyntaxWarning: invalid escape sequence '\L'
<>:10: SyntaxWarning: invalid escape sequence '\L'
<>:11: SyntaxWarning: invalid escape sequence '\L'
C:\Users\stutzmann\AppData\Local\Temp\ipykernel_27364\909450486.py:9: SyntaxWarning: invalid escape sequence '\L'
  JSON_PATH = "..\List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio.json"   # <- ton export Label Studio
C:\Users\stutzmann\AppData\Local\Temp\ipykernel_27364\909450486.py:10: SyntaxWarning: invalid escape sequence '\L'
  CSV_PATH  = "..\List-of-images\JJ096-JJ099_image_data.csv"                # <- ta liste complète d'images
C:\Users\stutzmann\AppData\Local\Temp\ipykernel_27364\909450486.py:11: SyntaxWarning: invalid escape sequence '\L'
  OUTPUT_PATH = "..\List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio_completed.json"


In [21]:
# ============================================================
# Cellule 3 (modifiée) — charger le CSV en dict pour accès par nom d'image
# ============================================================
CSV_IMAGE_COLUMN = "imageFileName"
CSV_URL_COLUMN = "urlImage"

csv_rows_by_image = {}
with open(CSV_PATH, "r", encoding="utf-7") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row[CSV_IMAGE_COLUMN].strip()
        if name:
            csv_rows_by_image[basename(name)] = row

csv_images = list(csv_rows_by_image.keys())
csv_images_sorted = sorted(csv_images, key=numeric_key)

In [22]:
# ============================================================
# Cellule 4 — Fonction d'extraction de la clé numérique (tri "naturel")
# ============================================================
# Si tes noms d'images ressemblent à "JJ167_f001r.jpg", "img_23.png", etc.,
# on extrait le(s) nombre(s) pour trier correctement (1, 2, 10 et non 1, 10, 2)

def numeric_key(name: str):
    filename = Path(name).name  # enlève le chemin/URL éventuel
    numbers = re.findall(r"\d+", filename)
    return [int(n) for n in numbers] if numbers else [float("inf")]

# on trie la liste CSV selon cette clé, pour établir l'ordre de référence
csv_images_sorted = sorted(csv_images, key=numeric_key)

# ============================================================
# Cellule 5 — Identifier les images manquantes dans le JSON
# ============================================================
def basename(path_or_url: str) -> str:
    return Path(path_or_url).name

existing_images = {basename(t["data"].get(IMAGE_FIELD, "")) for t in tasks}
missing_images = [img for img in csv_images_sorted if basename(img) not in existing_images]

print(f"Images manquantes ({len(missing_images)}) :")
for m in missing_images:
    print(" -", m)

Images manquantes (7) :
 - Paris_Archives_Nationales_JJ096_1.jpg
 - Paris_Archives_Nationales_JJ096_297.jpg
 - Paris_Archives_Nationales_JJ097_1.jpg
 - Paris_Archives_Nationales_JJ097_300.jpg
 - Paris_Archives_Nationales_JJ098_1.jpg
 - Paris_Archives_Nationales_JJ099_1.jpg
 - Paris_Archives_Nationales_JJ099_389.jpg


In [23]:
# ============================================================
# Cellule 6 (corrigée) — image_path = valeur brute du CSV, non réduite
# ============================================================
def make_new_task(image_key: str) -> dict:
    row = csv_rows_by_image[image_key]  # image_key = basename, sert juste à retrouver la ligne

    # --- valeur brute telle qu'elle apparaît dans le CSV ---
    full_image_path = row[CSV_IMAGE_COLUMN].strip()

    # --- champ "image" : url avec remplacement de "/full/full/" ---
    url_image = row.get(CSV_URL_COLUMN, "").strip()
    image_value = url_image.replace("/full/full/", "/full/1200,/")

    # --- registre / ordre à partir du image_path complet (sans extension) ---
    stem = Path(full_image_path.replace("\\", "/")).stem
    tokens = stem.split("_")
    ordre = tokens[-1] if len(tokens) >= 1 else None
    registre = tokens[-2] if len(tokens) >= 2 else None

    return {
        "id": None,
        "data": {
            "image": image_value,
            IMAGE_FIELD: full_image_path,   # <- valeur brute du CSV, telle quelle
            "registre": registre,
            "ordre": ordre
        },
        "annotations": [],
        "predictions": []
    }

new_tasks = [make_new_task(img) for img in missing_images]

# ============================================================
# Cellule 7 — Fusionner et trier l'ensemble selon l'ordre numérique
# ============================================================
all_tasks = tasks + new_tasks

all_tasks_sorted = sorted(
    all_tasks,
    key=lambda t: numeric_key(basename(t["data"].get(IMAGE_FIELD, "")))
)

print(f"Total après fusion : {len(all_tasks_sorted)} tâches")

Total après fusion : 1554 tâches


In [24]:
# ============================================================
# Cellule 8 — Sauvegarder le nouveau JSON d'import
# ============================================================
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_tasks_sorted, f, ensure_ascii=False, indent=2)

print(f"Fichier écrit : {OUTPUT_PATH}")

Fichier écrit : ..\List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio_completed.json
